In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Thin create-only handoff; the runner owns terminal ZIP and sidecar publication.
from datetime import datetime, timezone
import json, os, pathlib, re, subprocess, sys, uuid
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Content-Texture'
EXPECTED_EXACT = '7917a7da15fbeee79083b4938362d2bdf202a740'
RUNNER_MODULE = 'experiments.run_content_texture_stratification_v1'
CLAIM_CEILING = 'exploratory_prospective_texture_stratification_only'
ATTEMPT_NONCE = uuid.uuid4().hex[:12]
SOURCE = pathlib.Path(f'/content/cegwm-content-texture-{ATTEMPT_NONCE}-source')
LOCAL = pathlib.Path(f'/content/Content-Texture-7917a7d-{ATTEMPT_NONCE}-local')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
RUNNER_ATTEMPTED = False
CAPTURE_LIMIT = 4096


In [ ]:
if SOURCE.exists() or LOCAL.exists(): raise FileExistsError('create-only local path')
subprocess.run(['git', 'clone', '--no-single-branch', '--branch', BRANCH, REPO_URL, str(SOURCE)], check=True)
def git(*args): return subprocess.run(['git', *args], cwd=SOURCE, check=True, capture_output=True, text=True).stdout.strip()
HANDOFF_HEAD = git('rev-parse', 'HEAD')
if git('branch', '--show-current') != BRANCH or git('status', '--porcelain'): raise RuntimeError('canonical checkout identity')
if subprocess.run(['git','merge-base','--is-ancestor',EXPECTED_EXACT,HANDOFF_HEAD],cwd=SOURCE).returncode != 0: raise RuntimeError('execution exact is not handoff ancestor')
git('checkout', '--detach', EXPECTED_EXACT)
if git('branch', '--show-current') or git('rev-parse', 'HEAD') != EXPECTED_EXACT or git('status', '--porcelain'): raise RuntimeError('detached execution identity')
while True:
    RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); DRIVE_TARGET = DRIVE_ROOT / f'Content-Texture-7917a7d-{RUN_UTC}'
    if not DRIVE_TARGET.exists(): break
subprocess.run([sys.executable, '-m', 'pip', 'install', str(SOURCE)], check=True)
if git('branch', '--show-current') or git('rev-parse', 'HEAD') != EXPECTED_EXACT or git('status', '--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
env = {k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}
env['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY'); env['HF_TOKEN'] = userdata.get('HF_TOKEN')
RUNNER_ATTEMPTED = True
p = subprocess.Popen([sys.executable, '-m', RUNNER_MODULE, '--repo-root', str(SOURCE), '--expected-exact', EXPECTED_EXACT, '--local-work-root', str(LOCAL), '--artifact-sink', str(DRIVE_TARGET), '--provenance-root', '/content/drive/MyDrive/CEG-WM'], cwd=SOURCE, env=env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
env.pop('CEG_WM_ROOT_KEY', None); env.pop('HF_TOKEN', None); env = None
captured = p.stdout.read(CAPTURE_LIMIT + 1); rc = p.wait()
if len(captured) > CAPTURE_LIMIT or rc not in (0,2): raise RuntimeError('bounded runner result')


In [ ]:
# Sidecar/filename-only terminal validation; the notebook never reads or rehashes ZIP bytes.
line = captured.decode('utf-8', 'strict').strip(); prefix = 'CEGWM_TEXTURE_RESULT '
if not line.startswith(prefix): raise RuntimeError('sanitized result required')
result = json.loads(line[len(prefix):]); kind=result.get('artifact_kind'); status=result.get('status'); analysis_status=result.get('analysis_status')
if (rc,kind,status,analysis_status) == (2,'operational_terminal','operational_failure','operational_failure'):
    print('CEGWM_TEXTURE_HANDOFF_FAILURE ' + json.dumps({'execution_exact':EXPECTED_EXACT,'failure_class':result.get('failure_class'),'failure_stage':result.get('failure_stage'),'run_id':result.get('run_id')},sort_keys=True,separators=(',',':')))
elif (rc,kind,status,analysis_status) in ((0,'terminal','analysis_complete','analysis_complete'),(2,'terminal','not_interpretable','not_interpretable')):
    terminal = DRIVE_TARGET / EXPECTED_EXACT / result['run_id'] / 'terminal'; archive = terminal / (result['run_id'] + '.zip'); sidecar = terminal / (result['run_id'] + '.zip.sha256')
    if not archive.is_file() or not sidecar.is_file() or sidecar.stat().st_size > CAPTURE_LIMIT or sidecar.read_text('ascii') != result['terminal_sha256'] + '  ' + archive.name + '\n': raise RuntimeError('terminal pair')
    print('CEGWM_TEXTURE_ARTIFACT ' + json.dumps({'artifact_kind':kind,'analysis_status':analysis_status,'execution_exact':EXPECTED_EXACT,'run_id':result['run_id'],'archive_path':str(archive),'sidecar_path':str(sidecar)},sort_keys=True,separators=(',',':')))
else: raise RuntimeError('runner terminal status contract')
